In [3]:
# 匯入 ArcPy
import arcpy
from arcpy.sa import Raster

# 允許覆蓋輸出
arcpy.env.overwriteOutput = True

# 啟用 Spatial Analyst 擴充套件
arcpy.CheckOutExtension("Spatial")

# === 輸入路徑 ===
in_ras = r"D:\SfM_coral_reefs_product\Southern_Taiwan\Outlet\OL_P3\OL_P3_2204_3826_05mm_Clip.tif"    # 這是你的三波段正射影像
out_ras = r"D:\SfM_coral_reefs_product\Southern_Taiwan\Outlet\OL_P3\OL_P3_2204_3826_05mm_Clip_gray_m.tif"  # 輸出的單波段灰階檔案

# 讀取各個波段
b1 = Raster(in_ras + r"\Band_1")
b2 = Raster(in_ras + r"\Band_2")
b3 = Raster(in_ras + r"\Band_3")

# === 灰階計算方式 ===
# 方法一：取三個波段的平均
gray_m = (b1 + b2 + b3) / 3.0

# 方法二（可選）：加權平均（常見的灰階轉換公式）
#gray = 0.3 * b1 + 0.59 * b2 + 0.11 * b3

# === 輸出結果 ===
gray_m.save(out_ras)

print("處理完成！輸出檔案在：", out_ras)


處理完成！輸出檔案在： D:\SfM_coral_reefs_product\Southern_Taiwan\Outlet\OL_P3\OL_P3_2204_3826_05mm_Clip_gray_m.tif


In [11]:
import rasterio
import numpy as np

# 輸入檔案
in_file = r"D:\ArcGIS\Morpho_Spatial_Demography\OL_P3_2204_Compo_GSC.tif"
out_file = "OL_P3_2204_Compo_GSC_norm.tif"

# 讀取三個 band
with rasterio.open(in_file) as src:
    profile = src.profile.copy()   # 保存空間參考、dtype、影像大小等
    data = src.read()             # shape = (bands, height, width)
    # data[0,:,:] = band1 (灰階)
    # data[1,:,:] = band2 (Slope)
    # data[2,:,:] = band3 (Curvature)

# ========== 方法一：Min-Max 正規化到 [0,1] ==========
norm_data = np.zeros_like(data, dtype=np.float32)

for b in range(data.shape[0]):
    band = data[b].astype(np.float32)
    band_min, band_max = np.nanmin(band), np.nanmax(band)  # 忽略 NaN
    norm_data[b] = (band - band_min) / (band_max - band_min + 1e-9)  # 防止除0

# ========== 方法二：Z-score (可選) ==========
# zscore_data = np.zeros_like(data, dtype=np.float32)
# for b in range(data.shape[0]):
#     band = data[b].astype(np.float32)
#     mu, sigma = np.nanmean(band), np.nanstd(band)
#     zscore_data[b] = (band - mu) / (sigma + 1e-9)

# 更新 profile，確保是浮點輸出
profile.update(
    dtype=rasterio.float32,
    count=norm_data.shape[0]  # 三個 band
)

# 輸出新 tif
with rasterio.open(out_file, 'w', **profile) as dst:
    dst.write(norm_data)

print("✅ 正規化完成，輸出檔案:", out_file)
